In [ ]:
import copy
import math
import random
from typing import List, Tuple
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

# reproducibility
SEED = 12345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [ ]:
# def state_dict_to_list(sd: dict) -> List[np.ndarray]:
#     return [v.detach().cpu().numpy() for _, v in sd.items()]

# def list_to_state_dict(model: nn.Module, lst: List[np.ndarray]) -> dict:
#     sd = model.state_dict()
#     keys = list(sd.keys())
#     new = {}
#     for k, arr in zip(keys, lst):
#         new[k] = torch.tensor(arr, dtype=sd[k].dtype)
#     return new

def state_dict_to_list(state_dict):
    flat_list = []
    shapes = []
    for key, param in state_dict.items():
        arr = param.cpu().numpy().ravel()   # flatten
        flat_list.extend(arr)
        shapes.append((key, param.shape, len(arr)))
    return flat_list, shapes

def list_to_state_dict(flat_list, shapes, model):
    state_dict = {}
    idx = 0
    for key, shape, length in shapes:
        arr = flat_list[idx: idx + length]
        arr = np.array(arr).reshape(shape)
        state_dict[key] = torch.tensor(arr, dtype=model.state_dict()[key].dtype)
        idx += length
    return state_dict


def load_list_into(model: nn.Module, lst: List[np.ndarray]):
    sd_new = list_to_state_dict(model, lst)
    model.load_state_dict(sd_new, strict=True)

In [ ]:
def avg_aggregate(updates: List[Tuple[List[np.ndarray], int]]) -> Tuple[List[np.ndarray], int]:
    if len(updates) == 0:
        return [], 0
    weights = np.array([u[1] for u in updates], dtype=np.float64)
    if weights.sum() == 0:
        weights = np.ones_like(weights)
    w = weights / weights.sum()
    arrays_per_param = list(zip(*[u[0] for u in updates]))
    agg = []
    for param_vals in arrays_per_param:
        stacked = np.stack(param_vals, axis=0)
        agg_param = (w.reshape((-1,) + (1,) * (stacked.ndim - 1)) * stacked).sum(axis=0)
        agg.append(agg_param)
    total = int(weights.sum())
    return agg, total

In [ ]:
GLOBAL_HELPER_UPDATES = dict()

In [ ]:
class Client:
    def __init__(self, cid: int, dataset: Subset, device='cpu'):
        self.cid = cid
        self.dataset = dataset
        self.device = device

    def local_train(self, model: nn.Module, epochs=1, batch_size=32, lr=0.01):
        loader = DataLoader(self.dataset, batch_size=batch_size, shuffle=True)
        local_model = copy.deepcopy(model).to(self.device)
        opt = optim.SGD(local_model.parameters(), lr=lr, momentum=0.9)
        loss_fn = nn.CrossEntropyLoss()
        local_model.train()
        for _ in range(epochs):
            for X, y in loader:
                X, y = X.to(self.device), y.to(self.device)
                opt.zero_grad()
                loss_fn(local_model(X), y).backward()
                opt.step()
        # return weights (post-training) and number of samples
        # print(local_model)
        print("model", local_model)
        weights = local_model.state_dict()
        print("wts",weights)
        num_samples = len(loader.dataset)
        mid = len(weights) // 2
        first_half = weights[:mid]
        second_half = weights[mid:]
        third_entry = [a+b for a,b in zip(first_half,second_half)]
        ret_list = [first_half, second_half, third_entry]
        drop_num = None
        rand_num = random.randint(0,10) 
        if rand_num<3:
            drop_num = rand_num 
        if drop_num:
            ret_list[drop_num]=None
        GLOBAL_HELPER_UPDATES[self.cid]=(ret_list,num_samples)
        # GLOBAL_CLIENT_UPDATES[self.cid] = (weights,)
        
        return state_dict_to_list(local_model.state_dict()), len(loader.dataset)

In [ ]:
print()

In [ ]:
class Helper:
    """
    Helper (edge) that collects raw updates from assigned clients and forwards them to master.
    It does NOT aggregate — it simply forwards the subset of client updates that arrived.
    """
    def __init__(self, hid: int, clients: List[Client], device='cpu'):
        self.hid = hid
        self.clients = clients  # list of Client objects (a client may appear in multiple helpers if replicated)
        self.device = device

    def collect_updates(self, model: nn.Module, client_epochs=1, client_batch=32, client_lr=0.01, drop_prob=0.5):
        """
        Ask each assigned client to train and forward its update.
        drop_prob: probability that the helper fails to receive that client's update (simulating straggling/drops).
        Returns: list of tuples (update_list, num_samples, client_id, helper_id)
        """
        client_updates = []
        curr_clients = [self.clients[self.hid],[self.hid+1]]
        for client in curr_clients:
            # simulate unreliable connection
            if random.random() < drop_prob:
                # dropped: do not include this client's update in forwarded list
                continue
            update, num_samples = client.local_train(model, epochs=client_epochs, batch_size=client_batch, lr=client_lr)
            
            client_updates.append((update, num_samples, client.cid, self.hid))
        return client_updates

In [ ]:
class MasterServer:
    def __init__(self, model: nn.Module, helpers: List[Helper], device="cpu"):
        self.model = model
        self.helpers = helpers
        self.device = device

    def run_round(
        self, client_epochs=1, client_batch=32, client_lr=0.01, drop_prob=0.2
    ):
        """
        For each helper:
          - helper.collect_updates(...) returns a list of raw client updates it received (some dropped randomly)
        Master receives all helper-forwarded updates, resolves duplicates (one update per client),
        then aggregates (FedAvg over unique client updates).
        """
        all_updates = []
        for helper in self.helpers:
            updates = helper.collect_updates(
                self.model, client_epochs, client_batch, client_lr, drop_prob=drop_prob
            )
            all_updates.extend(updates)
        
        print(GLOBAL_HELPER_UPDATES[0])

        unique_updates = {}
        for update, num_samples, cid, hid in all_updates:
            if cid not in unique_updates:
                unique_updates[cid] = (
                    update,
                    num_samples,
                    hid,
                )  # store helper id for debugging if needed

        def reconstruct_update(coded_update):
            part1, part2, part3 = coded_update
            if part1 is None:
                part1 = [p3 - p2 for p2, p3 in zip(part2, part3)]
            elif part2 is None:
                part2 = [p3 - p1 for p1, p3 in zip(part1, part3)]
            elif part3 is None:
                part3 = [p1 + p2 for p1, p2 in zip(part1, part2)]
            return part1 + part2
        
        if len(unique_updates) == 0:
            return
        reconstruct_updates  = []
        for client_update, num_samples in GLOBAL_HELPER_UPDATES:
            reconstruct_updates.append((reconstruct_update(client_update),num_samples))

        agg_state, _ = avg_aggregate(reconstruct_updates)

        load_list_into(self.model, agg_state)

    def evaluate(self, test_loader: DataLoader):
        self.model.eval()
        correct = 0
        total = 0
        loss_fn = nn.CrossEntropyLoss()
        loss_sum = 0.0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(self.device), y.to(self.device)
                logits = self.model(X)
                loss_sum += loss_fn(logits, y).item()
                preds = logits.argmax(dim=1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        return loss_sum / len(test_loader), correct / total

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

num_clients = 12
num_helpers = 3
replication_factor = 2  
drop_prob = 0.2         
client_len = len(train_dataset) // num_clients

client_subsets = [Subset(train_dataset, list(range(i * client_len, (i + 1) * client_len)))
                  for i in range(num_clients)]
clients = [Client(cid=i, dataset=client_subsets[i]) for i in range(num_clients)]


helpers_clients = [[] for _ in range(num_helpers)]
for cid, client in enumerate(clients):

    assigned = [(cid + r) % num_helpers for r in range(replication_factor)]
    for h in assigned:
        helpers_clients[h].append(client)

helpers = [Helper(hid=i, clients=helpers_clients[i]) for i in range(num_helpers)]

master_model = SimpleNet()
master = MasterServer(model=master_model, helpers=helpers)

In [ ]:
ROUNDS = 10
global_accuracies = []
losses = []

for r in range(1, ROUNDS + 1):
    print(f"--- Round {r} ---")
    master.run_round(client_epochs=1, client_batch=32, client_lr=0.05, drop_prob=drop_prob)
    loss, acc = master.evaluate(test_loader)
    print(f"Test Loss: {loss:.4f}, Test Accuracy: {acc*100:.2f}%")
    global_accuracies.append(acc)
    losses.append(loss)

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, ROUNDS + 1), global_accuracies, marker='o')
plt.title("Global Model Accuracy")
plt.xlabel("Round")
plt.ylabel("Accuracy")
plt.subplot(1, 2, 2)
plt.plot(range(1, ROUNDS + 1), losses, marker='o', color='r')
plt.title("Global Model Loss")
plt.xlabel("Round")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()